# WorkPulse – Unemployment Trends & Economic Insights
### CodeAlpha Data Science Internship Task 2
**Author:** Niviya Albert | **Role:** Data Science Intern

---
## 🎯 Project Overview & Business Problem
Unemployment is one of the most critical macroeconomic indicators reflecting economic vitality, labor demand, and social well-being. The COVID-19 pandemic and subsequent national lockdowns in 2020 induced sudden, severe economic disruptions.

### Objectives:
1. **Baseline Assessment:** Establish pre-COVID unemployment benchmarks (May 2019 – Feb 2020).
2. **COVID-19 Shock Impact:** Measure peak unemployment and job loss during strict lockdown (April–May 2020).
3. **Rural vs. Urban Dynamics:** Compare the resilience and recovery velocity between rural agricultural and urban service/industrial economies.
4. **Geographic & Zonal Disparities:** Identify vulnerable states requiring targeted labor and welfare policies.
5. **Policy Recommendations:** Provide data-backed actionable interventions for economic stability.

In [ ]:
# Step 1: Environment Setup & Library Imports
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set modern aesthetics
sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 150
plt.rcParams['figure.figsize'] = (12, 6)

print('Environment initialized successfully.')

In [ ]:
# Step 2: Load Preprocessed Datasets
df_primary = pd.read_csv('../data/cleaned_unemployment_data.csv')
df_primary['Date'] = pd.to_datetime(df_primary['Date'])

df_regional = pd.read_csv('../data/cleaned_unemployment_regional.csv')
df_regional['Date'] = pd.to_datetime(df_regional['Date'])

print(f'Primary Dataset Shape: {df_primary.shape} | Dates: {df_primary["Date"].min().date()} to {df_primary["Date"].max().date()}')
print(f'Regional Dataset Shape: {df_regional.shape} | Dates: {df_regional["Date"].min().date()} to {df_regional["Date"].max().date()}')
df_primary.head()

## 📈 Section 1: National Unemployment Trend Over Time
Let us examine how national unemployment evolved from pre-COVID normal (May 2019) through the lockdown spike (April–May 2020) and subsequent reopening.

In [ ]:
# Monthly Aggregations
monthly_national = df_primary.groupby('Date')['Unemployment_Rate_Pct'].agg(['mean', 'median', 'std', 'min', 'max']).reset_index()
monthly_rural = df_primary[df_primary['Area_Type'] == 'Rural'].groupby('Date')['Unemployment_Rate_Pct'].mean().reset_index()
monthly_urban = df_primary[df_primary['Area_Type'] == 'Urban'].groupby('Date')['Unemployment_Rate_Pct'].mean().reset_index()

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(monthly_national['Date'], monthly_national['mean'], marker='o', color='#0F172A', linewidth=3, label='National Average')
ax.plot(monthly_urban['Date'], monthly_urban['Unemployment_Rate_Pct'], marker='s', linestyle='--', color='#E11D48', linewidth=2, label='Urban Average')
ax.plot(monthly_rural['Date'], monthly_rural['Unemployment_Rate_Pct'], marker='^', linestyle='--', color='#16A34A', linewidth=2, label='Rural Average')

# Shading lockdown shock
ax.axvspan(pd.Timestamp('2020-03-24'), pd.Timestamp('2020-05-31'), color='#FEE2E2', alpha=0.6, label='Strict Lockdown Phase')

ax.set_title('National Unemployment Trajectory: Rural vs. Urban (May 2019 – June 2020)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Timeline', fontweight='bold')
ax.set_ylabel('Unemployment Rate (%)', fontweight='bold')
ax.legend(frameon=True, facecolor='white', edgecolor='#CBD5E1')
plt.tight_layout()
plt.show()

## 📍 Section 2: Regional & State-Wise Analysis
Which states experienced the highest chronic unemployment, and which states maintained resilient employment?

In [ ]:
state_ranking = df_primary.groupby('State')['Unemployment_Rate_Pct'].mean().sort_values(ascending=False)
top_10 = state_ranking.head(10)
bottom_10 = state_ranking.tail(10).sort_values(ascending=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(x=top_10.values, y=top_10.index, ax=ax1, hue=top_10.index, palette='Reds_r', legend=False)
ax1.set_title('Top 10 States with Highest Avg Unemployment (%)', fontweight='bold')
ax1.set_xlabel('Average Unemployment Rate (%)')
for p in ax1.patches:
    ax1.annotate(f'{p.get_width():.1f}%', (p.get_width() + 0.5, p.get_y() + p.get_height() / 2), ha='left', va='center', fontsize=9, fontweight='bold')

sns.barplot(x=bottom_10.values, y=bottom_10.index, ax=ax2, hue=bottom_10.index, palette='Greens_d', legend=False)
ax2.set_title('Top 10 States with Lowest Avg Unemployment (%)', fontweight='bold')
ax2.set_xlabel('Average Unemployment Rate (%)')
for p in ax2.patches:
    ax2.annotate(f'{p.get_width():.1f}%', (p.get_width() + 0.3, p.get_y() + p.get_height() / 2), ha='left', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 🦠 Section 3: Dedicated COVID-19 Impact & Recovery Analysis
Let us quantitatively compare the Pre-COVID baseline vs. COVID Lockdown vs. Reopening/Recovery periods.

In [ ]:
# Categorize periods
def categorize_period(date):
    if date < pd.Timestamp('2020-03-01'):
        return 'Pre-COVID (May 2019 - Feb 2020)'
    elif date <= pd.Timestamp('2020-05-31'):
        return 'Lockdown Shock (Mar - May 2020)'
    else:
        return 'Reopening / Recovery (Jun 2020+)'

df_primary['COVID_Period'] = df_primary['Date'].apply(categorize_period)

covid_summary = df_primary.groupby('COVID_Period')['Unemployment_Rate_Pct'].agg(['count', 'mean', 'std', 'min', 'max']).reset_index()
print('=== COVID PERIOD COMPARISON ===')
display(covid_summary)

# Period comparison boxplot
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_primary, x='COVID_Period', y='Unemployment_Rate_Pct', palette='Set2')
plt.title('Unemployment Rate Distribution by COVID Period Phase', fontweight='bold', pad=15)
plt.ylabel('Unemployment Rate (%)', fontweight='bold')
plt.xlabel('')
plt.tight_layout()
plt.show()

## 🔥 Section 4: State-Wise Monthly Heatmap
Visualizing the exact month-by-month unemployment intensity across every Indian state.

In [ ]:
pivot_df = df_primary.pivot_table(index='State', columns='Year_Month', values='Unemployment_Rate_Pct', aggfunc='mean')
pivot_df = pivot_df.loc[state_ranking.index]

plt.figure(figsize=(14, 10))
sns.heatmap(pivot_df, cmap='YlOrRd', annot=True, fmt='.1f', linewidths=0.5, cbar_kws={'label': 'Unemployment Rate (%)'})
plt.title('State-Wise Monthly Unemployment Rate Heatmap (2019 - 2020)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Year-Month', fontweight='bold')
plt.ylabel('State', fontweight='bold')
plt.tight_layout()
plt.show()

## 🧭 Section 5: Macro-Zonal Analysis & Extended Recovery (Jan 2020 – Oct 2020)

In [ ]:
zone_monthly = df_regional.groupby(['Year_Month', 'Zone'])['Unemployment_Rate_Pct'].mean().unstack()

plt.figure(figsize=(12, 6))
for zone in zone_monthly.columns:
    plt.plot(zone_monthly.index, zone_monthly[zone], marker='o', linewidth=2.5, label=zone)

plt.title('Zonal Unemployment Trajectory & Extended Recovery (Jan 2020 – Oct 2020)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Year-Month', fontweight='bold')
plt.ylabel('Unemployment Rate (%)', fontweight='bold')
plt.legend(title='Macro-Zone', frameon=True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()